# Pipeline RAG completo

Este é o notebook principal (equivalente ao `rag.py`). Ele lê os PDFs de `documents/`, cria chunks com overlap, persiste embeddings no ChromaDB e compara os dois métodos pedidos no `TODO.md`:

- **esparso:** TF-IDF com similaridade cosseno;
- **denso:** embeddings Gemini com similaridade cosseno no ChromaDB.

Ao final, três perguntas são respondidas sem retrieval e com os dois retrievers. Uma pergunta demonstrativa adicional completa as 10 respostas solicitadas pela ficha de avaliação manual.

## 1. Modelos, caminhos e banco

In [9]:
import os
from pathlib import Path
from pprint import pprint

import chromadb
from dotenv import load_dotenv
from google import genai

import rag_utils

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "documents"
CHROMA_DIR = PROJECT_ROOT / "db" / "chroma"
COLLECTION_NAME = "rag_documents_v1"
EMBED_MODEL = "gemini-embedding-001"
GEN_MODEL = "gemini-3.1-flash-lite"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
TOP_K = 5

load_dotenv(PROJECT_ROOT / ".env")
API_KEY = os.environ.get("GEMINI_API_KEY")
assert API_KEY, "Defina GEMINI_API_KEY no arquivo .env da raiz do projeto."

client = genai.Client(api_key=API_KEY)
embedder = rag_utils.GeminiEmbedder(client, model=EMBED_MODEL)
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"}
)
print(f"Documentos: {DATA_DIR}")
print(f"Banco: {CHROMA_DIR}")

Documentos: /home/joaomendonca/rag-practice/documents
Banco: /home/joaomendonca/rag-practice/db/chroma


## 2. Ingestão e indexação

A indexação é retomável: chunks já presentes no banco não são enviados novamente à API de embeddings.

In [10]:
pdf_paths = rag_utils.iter_pdfs(DATA_DIR)
records = rag_utils.load_corpus(DATA_DIR, CHUNK_SIZE, CHUNK_OVERLAP)
assert records, f"Nenhum conteúdo PDF foi encontrado em {DATA_DIR}."

index_stats = rag_utils.index_dense_collection(collection, records, embedder)
print(f"PDFs: {len(pdf_paths)}")
print(f"Chunks: {len(records)}")
print(index_stats)

PDFs: 4
Chunks: 94
{'total': 94, 'already_indexed': 0, 'indexed_now': 94}


## 3. Os dois métodos de retrieval

Ambos usam exatamente os mesmos chunks. O método denso consulta os vetores persistidos no Chroma; o TF-IDF monta um índice esparso local a partir dos textos guardados no mesmo banco.

In [11]:
indexed_records = rag_utils.records_from_collection(collection)
dense_retriever = rag_utils.DenseRetriever(collection, embedder)
tfidf_retriever = rag_utils.TfidfRetriever(indexed_records)

dense_rag = rag_utils.RAGAssistant(client, dense_retriever, GEN_MODEL)
tfidf_rag = rag_utils.RAGAssistant(client, tfidf_retriever, GEN_MODEL)

In [12]:
sample_question = "Quais animais já foram votados para cargos políticos?"

print("Ranking denso:")
pprint(rag_utils.hits_as_rows(dense_retriever.retrieve(sample_question, TOP_K)), sort_dicts=False)
print("\nRanking TF-IDF:")
pprint(rag_utils.hits_as_rows(tfidf_retriever.retrieve(sample_question, TOP_K)), sort_dicts=False)

Ranking denso:
[{'rank': 1,
  'cosine_similarity': 0.7154,
  'paper': 'Rinoceronte_Cacareco',
  'page': 2,
  'chunk': 0,
  'snippet': 'Macaco Tião\n'
             'Bode Ioiô\n'
             '1. Rookmaaker, L.C.; Jones, Marvin L.; Klös, Heinz-Georg; '
             'Reynolds III, Richard J. (1998). The\n'
             'rhinoceros in captivity: a list of 2439 rhinoceroses kept from '
             'Roman times to 1994. Haia,\n'
             'Países Baixos…'},
 {'rank': 2,
  'cosine_similarity': 0.7141,
  'paper': 'Macaco_Tião',
  'page': 2,
  'chunk': 0,
  'snippet': 'teve a manchete "O prefeito do Rio, Júlio Coutinho, foi agredido '
             'hoje à tarde durante uma solenidade de\n'
             'inauguração" narrada por Sérgio Chapelin; o prefeito, ao ser '
             'indagado o motivo da agressão, respondeu à\n'
             'repórter Fer…'},
 {'rank': 3,
  'cosine_similarity': 0.7126,
  'paper': 'Rinoceronte_Cacareco',
  'page': 1,
  'chunk': 0,
  'snippet': 'Cacareco\n'
      

## 4. Geração fundamentada

O prompt contém os top-k chunks completos e exige citações no formato `[documento, p. N, trecho M]`.

In [13]:
demo_result = dense_rag.ask(sample_question, k=TOP_K)
print(demo_result.answer)

Com base nos documentos fornecidos, os animais que foram votados para cargos políticos são:

*   **Cacareco:** Uma rinoceronte fêmea que, nas eleições municipais de outubro de 1959 em São Paulo, recebeu cerca de 100 mil votos para vereador como forma de protesto [Rinoceronte_Cacareco, p. 1, trecho 0].
*   **Macaco Tião:** Teve sua candidatura à Prefeitura do Rio de Janeiro lançada em 1988 pela revista humorística *Casseta Popular*, recebendo 400 mil votos como voto de protesto [Macaco_Tião, p. 2, trecho 0].
*   **Stubbs (gato):** Foi eleito prefeito de uma cidade no Alasca após moradores, insatisfeitos com os candidatos, incentivarem outros a votarem nele como um candidato *write-in* (candidato escrito pelo eleitor) [Stubbs_(cat), p. 3, trecho 0].


## 5. Três perguntas: sem retrieval, RAG denso e RAG TF-IDF

Esta célula gera nove respostas comparáveis. Somadas à demonstração acima, são as 10 respostas usadas na avaliação manual.

In [14]:
QUESTIONS = [
    "Quais animais ja forma votados para cargos politicos no brasil?",
    "Quais animais dos documentos já morreram?",
    "Quais animais viram memes na internet?",
]

generated_answers = [
    {"question": sample_question, "method": "dense", "answer": demo_result.answer}
]

for question in QUESTIONS:
    baseline = rag_utils.answer_without_retrieval(client, question, GEN_MODEL)
    dense_result = dense_rag.ask(question, k=TOP_K)
    tfidf_result = tfidf_rag.ask(question, k=TOP_K)

    generated_answers.extend([
        {"question": question, "method": "no_retrieval", "answer": baseline},
        {"question": question, "method": "dense", "answer": dense_result.answer},
        {"question": question, "method": "tfidf", "answer": tfidf_result.answer},
    ])

    print(f"\n{'=' * 90}\nPERGUNTA: {question}")
    print(f"\nSEM RETRIEVAL\n{baseline}")
    print("\nCHUNKS — DENSO")
    pprint(rag_utils.hits_as_rows(dense_result.hits), sort_dicts=False)
    print(f"\nRAG DENSO\n{dense_result.answer}")
    print("\nCHUNKS — TF-IDF")
    pprint(rag_utils.hits_as_rows(tfidf_result.hits), sort_dicts=False)
    print(f"\nRAG TF-IDF\n{tfidf_result.answer}")

assert len(generated_answers) == 10


PERGUNTA: Quais animais ja forma votados para cargos politicos no brasil?

SEM RETRIEVAL
No Brasil, o uso de animais em eleições é um fenômeno conhecido como **"voto de protesto"** ou voto nulo simbólico. Como a legislação eleitoral brasileira exige que os candidatos sejam seres humanos (com CPF, título de eleitor e filiação partidária), os animais nunca foram eleitos oficialmente, mas suas candidaturas foram registradas de forma satírica ou simbólica.

Os casos mais famosos são:

### 1. Cacareco (Rinoceronte) – São Paulo, 1959
Este é o caso mais emblemático da história política brasileira. O rinoceronte do Zoológico de São Paulo foi lançado como candidato a vereador pelo jornalista Itaboraí Martins como uma forma de protesto contra a corrupção e a má qualidade dos candidatos da época.
*   **Resultado:** Cacareco recebeu cerca de **100 mil votos**, tornando-se o candidato mais votado daquela eleição. Como a candidatura não era oficial, os votos foram contabilizados como nulos, mas o i

## 6. Avaliação manual das 10 respostas

Preencha `True`/`False` depois de ler cada resposta e, nos casos com retrieval, os chunks exibidos. Para a baseline sem retrieval, suporte e citação não se aplicam.

In [ ]:
evaluation_sheet = [
    {
        "answer_id": index,
        "question": item["question"],
        "method": item["method"],
        "correct": None,
        "supported_by_chunks": None if item["method"] != "no_retrieval" else "N/A",
        "cites_right_chunk": None if item["method"] != "no_retrieval" else "N/A",
        "notes": "",
    }
    for index, item in enumerate(generated_answers, start=1)
]
evaluation_sheet

## 7. Diagnóstico de duas respostas erradas

Depois da avaliação, escolha dois `answer_id` incorretos. Marque **retrieval** se os chunks necessários não chegaram ao modelo; marque **generation** se os chunks eram adequados, mas a resposta os interpretou mal, ignorou evidências ou citou o trecho errado.

In [ ]:
failure_analysis = [
    {"answer_id": None, "failure_source": "retrieval ou generation", "justification": ""},
    {"answer_id": None, "failure_source": "retrieval ou generation", "justification": ""},
]
failure_analysis

## 8. Risco de busca responsável

**Atualidade:** o índice é uma fotografia dos PDFs no momento da ingestão; uma resposta pode parecer atual mesmo quando o corpus está desatualizado. **Mitigação:** registrar e mostrar a data e as fontes do corpus, reindexar periodicamente e invalidar chunks quando o documento de origem mudar.